# Home Credit Default Risk — Construcción del tablón en SQL (DuckDB)

Traducción a SQL/DuckDB de la lógica **oficial** de construcción de la base de modelización
(`HomeCredit_Construcción_de_la_base_de_datos_para_modelizar.ipynb`, versión pandas).

**Objetivo:** al ejecutarse contra los 8 CSV originales de Kaggle, reproducir exactamente `HomeCredit_base.csv`:
307,511 filas × 148 columnas (122 de `application_train` + 8 Bureau + 16 historial Home Credit + 2 indicadores de cobertura).

| Parte | Contenido |
|---|---|
| 1 | Carga de las tablas originales a `data/home_credit.duckdb` |
| 2 | Entorno SQL (`%sql` / `%%sql`) |
| 3 | Construcción del tablón (`tablon_general`) |
| 4 | Validación interna del tablón |
| 5 | Validación contra el tablón oficial `HomeCredit_base.csv` |
| 6 | Exportación |
| Anexo | Variables propias de la versión anterior (pendiente de decisión) |

## 0. Dependencias
Descomenta la línea si algún paquete no está instalado.

In [1]:
# %pip install duckdb jupysql duckdb-engine pandas

## 1. Carga de las tablas (un nombre por tabla)
Rutas: el notebook vive en `<repo>/sql`; la base DuckDB va a `data/home_credit.duckdb` (o a `HOMECREDIT_DB_PATH`). Los CSV de Kaggle **no se versionan**; se buscan (recursivamente) en `RAW_DIR`:
variable de entorno `HOMECREDIT_RAW_DIR` → `<repo>/data/raw` → carpeta local de la Tarea 1.

In [2]:
import os
from pathlib import Path

import duckdb
import pandas as pd

REPO_DIR = Path.cwd().parent if Path.cwd().name == "sql" else Path.cwd()
DB_PATH = Path(os.environ.get("HOMECREDIT_DB_PATH", REPO_DIR / "data" / "home_credit.duckdb"))   # base local (no se versiona)
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
(REPO_DIR / "data").mkdir(exist_ok=True)

_candidatas = [os.environ.get("HOMECREDIT_RAW_DIR"), REPO_DIR / "data" / "raw",
               REPO_DIR.parent / "Tareas Académicas" / "Tarea 1"]
RAW_DIR = next(Path(c) for c in _candidatas if c and Path(c).exists())


def ubicar(nombre):
    # Devuelve la ruta del archivo `nombre` dentro de RAW_DIR (búsqueda recursiva) o None.
    hallados = sorted(RAW_DIR.rglob(nombre))
    return hallados[0] if hallados else None


# nombre de la tabla -> ubicación del CSV
NOMBRES = ["application_train", "application_test", "bureau", "bureau_balance", "previous_application",
           "installments_payments", "credit_card_balance", "POS_CASH_balance"]
TABLAS_CSV = {n: ubicar(f"{n}.csv") for n in NOMBRES}
RUTA_DICCIONARIO = ubicar("HomeCredit_columns_description.csv")

faltantes = [n for n, p in {**TABLAS_CSV, "HomeCredit_columns_description": RUTA_DICCIONARIO}.items() if p is None]
assert not faltantes, f"No se encontraron estos archivos en {RAW_DIR}: {faltantes}"
print("Carpeta de datos crudos:", os.path.relpath(RAW_DIR, REPO_DIR))
print("Todos los archivos fueron encontrados.")

Carpeta de datos crudos: ../Tareas Académicas/Tarea 1
Todos los archivos fueron encontrados.


Se cargan los CSV **una sola vez** a `home_credit.duckdb` (si la tabla ya existe, se omite).
`MEMORIA_DUCKDB` (opcional, variable de entorno `DUCKDB_MEMORY_LIMIT`) limita la RAM que usa DuckDB.

In [3]:
con = duckdb.connect(str(DB_PATH))
MEMORIA_DUCKDB = os.environ.get("DUCKDB_MEMORY_LIMIT")      # p. ej. "2GB"; None = valor por defecto de DuckDB
if MEMORIA_DUCKDB:
    con.execute(f"SET memory_limit = '{MEMORIA_DUCKDB}'")

existentes = {r[0] for r in con.execute("SHOW TABLES").fetchall()}

for nombre, ruta in TABLAS_CSV.items():
    if nombre in existentes:
        print(f"[omitida] {nombre}: ya existe en la base")
        continue
    print(f"[cargando] {nombre} ...")
    # sample_size=-1: DuckDB revisa todo el archivo para inferir bien los tipos de dato
    con.execute(f'''
        CREATE TABLE {nombre} AS
        SELECT * FROM read_csv_auto('{ruta.as_posix()}', header=true, sample_size=-1)
    ''')

# Diccionario de columnas de Kaggle (latin-1, separado por ";" y con una columna índice sin nombre)
if "HomeCredit_columns_description" not in existentes:
    dic = pd.read_csv(RUTA_DICCIONARIO, encoding="latin-1", sep=";")
    dic = dic.loc[:, ~dic.columns.str.startswith("Unnamed")]
    con.register("_dic_tmp", dic)
    con.execute("CREATE TABLE HomeCredit_columns_description AS SELECT * FROM _dic_tmp")
    con.unregister("_dic_tmp")

# Resumen de lo cargado
resumen = [(t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]) for (t,) in con.execute("SHOW TABLES").fetchall()]
pd.DataFrame(resumen, columns=["tabla", "filas"])

[cargando] application_train ...


[cargando] application_test ...


[cargando] bureau ...


[cargando] bureau_balance ...


[cargando] previous_application ...


[cargando] installments_payments ...


[cargando] credit_card_balance ...


[cargando] POS_CASH_balance ...


,tabla,filas
0,HomeCredit_columns_description,219
1,POS_CASH_balance,10001358
2,application_test,48744
3,application_train,307511
4,bureau,1716428
5,bureau_balance,27299925
6,credit_card_balance,3840312
7,installments_payments,13605401
8,previous_application,1670214


Comprobación previa (celda 6 del notebook oficial): `application_train` tiene una fila por `SK_ID_CURR` y no comparte
`SK_ID_CURR` con `application_test` (test se usa solo para esta comprobación).

In [4]:
n_train, n_train_unicos = con.execute("SELECT COUNT(*), COUNT(DISTINCT SK_ID_CURR) FROM application_train").fetchone()
n_solapados, = con.execute('''
    SELECT COUNT(*) FROM application_train a JOIN application_test t USING (SK_ID_CURR)
''').fetchone()
print(f"SK_ID_CURR en train: {n_train:,} (únicos: {n_train_unicos:,}) | compartidos con test: {n_solapados}")
assert n_train == n_train_unicos and n_solapados == 0

SK_ID_CURR en train: 307,511 (únicos: 307,511) | compartidos con test: 0


## 2. Entorno SQL
Se activa la extensión `%sql` / `%%sql` (jupysql) conectada a la misma base.

In [5]:
%load_ext sql
%config SqlMagic.displaylimit = 10
%config SqlMagic.autopandas = False
%sql con

## 3. Tablón general: construcción
Misma lógica que el notebook oficial: **cada tabla histórica se agrega por separado a una fila por `SK_ID_CURR`**
y luego se hace `LEFT JOIN` con `application_train` (sin *mega-merge*).

| Bloque | Filtro oficial | Variables (orden oficial) |
|---|---|---|
| `bureau` + `bureau_balance` | créditos de train; meses `MONTHS_BALANCE < 0` | `BUREAU_N_CREDITOS`, `BUREAU_N_ACTIVOS`, `BUREAU_DEUDA_TOTAL`, `BUREAU_MAX_DIAS_ATRASO`, `BUREAU_MESES_OBSERVADOS`, `BUREAU_MESES_CON_ATRASO`, `BUREAU_MESES_ESTADO_CONOCIDO`, `BUREAU_PROP_MESES_CON_ATRASO` |
| `previous_application` | créditos de train | `HC_N_SOLICITUDES`, `HC_N_APROBADAS`, `HC_N_RECHAZADAS`, `HC_DIAS_ULTIMA_DECISION`, `HC_PROP_SOLICITUDES_RECHAZADAS` |
| `POS_CASH_balance` | `MONTHS_BALANCE < 0` | `HC_N_OPERACIONES_POS`, `HC_POS_MAX_ATRASO`, `HC_POS_MAX_CUOTAS_PENDIENTES` |
| `credit_card_balance` | `MONTHS_BALANCE < 0` | `HC_N_OPERACIONES_TARJETA`, `HC_CARD_MAX_ATRASO`, `HC_CARD_MAX_UTILIZACION`, `HC_CARD_MAX_ULTIMO_SALDO` |
| `installments_payments` | `DAYS_INSTALMENT < 0` (se conservan filas sin fecha de pago) | `HC_N_REGISTROS_PAGO`, `HC_N_PAGOS_TARDE`, `HC_PAY_MAX_ATRASO`, `HC_PROP_PAGOS_TARDE` |
| Cobertura | — | `TIENE_BUREAU`, `TIENE_HISTORIAL_HOME_CREDIT` |

Equivalencias pandas → SQL: `sum(min_count=1)` = `SUM` (NULL si todo es NULL); `x / y.replace(0, NaN)` = `x / NULLIF(y, 0)`;
"último mes por operación" (`sort_values` + `tail(1)`) = `ROW_NUMBER()` por `SK_ID_PREV` ordenado por `MONTHS_BALANCE DESC`
(empates → la última fila del archivo, `rowid DESC`, igual que el orden estable de pandas).

In [6]:
%%sql

CREATE OR REPLACE TABLE tablon_general AS
WITH ids_train AS (
    SELECT SK_ID_CURR FROM application_train
),
-- ── Bureau ───────────────────────────────────────────────────────────────────────────
bureau_train AS (                 -- créditos externos de los créditos objetivo de train
    SELECT * FROM bureau WHERE SK_ID_CURR IN (SELECT SK_ID_CURR FROM ids_train)
),
bb_por_credito AS (               -- nivel SK_ID_BUREAU (solo meses anteriores: MONTHS_BALANCE < 0)
    SELECT SK_ID_BUREAU,
           COUNT(*) AS BB_MESES_OBSERVADOS,
           SUM(CASE WHEN TRIM(STATUS) IN ('1','2','3','4','5')     THEN 1 ELSE 0 END)::BIGINT AS BB_MESES_CON_ATRASO,
           SUM(CASE WHEN TRIM(STATUS) IN ('0','1','2','3','4','5') THEN 1 ELSE 0 END)::BIGINT AS BB_MESES_ESTADO_CONOCIDO
    FROM bureau_balance
    WHERE MONTHS_BALANCE < 0
    GROUP BY SK_ID_BUREAU
),
bureau_agg AS (                   -- nivel SK_ID_CURR
    SELECT b.SK_ID_CURR,
           COUNT(DISTINCT b.SK_ID_BUREAU)                            AS BUREAU_N_CREDITOS,
           SUM(CASE WHEN b.CREDIT_ACTIVE = 'Active' THEN 1 ELSE 0 END)::BIGINT AS BUREAU_N_ACTIVOS,
           SUM(b.AMT_CREDIT_SUM_DEBT)                                AS BUREAU_DEUDA_TOTAL,
           MAX(b.CREDIT_DAY_OVERDUE)                                 AS BUREAU_MAX_DIAS_ATRASO,
           SUM(bb.BB_MESES_OBSERVADOS)::BIGINT                       AS BUREAU_MESES_OBSERVADOS,
           SUM(bb.BB_MESES_CON_ATRASO)::BIGINT                       AS BUREAU_MESES_CON_ATRASO,
           SUM(bb.BB_MESES_ESTADO_CONOCIDO)::BIGINT                  AS BUREAU_MESES_ESTADO_CONOCIDO
    FROM bureau_train b
    LEFT JOIN bb_por_credito bb USING (SK_ID_BUREAU)
    GROUP BY b.SK_ID_CURR
),
-- ── previous_application ─────────────────────────────────────────────────────────────
prev_agg AS (
    SELECT SK_ID_CURR,
           COUNT(DISTINCT SK_ID_PREV)                                          AS HC_N_SOLICITUDES,
           SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 ELSE 0 END)::BIGINT AS HC_N_APROBADAS,
           SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Refused'  THEN 1 ELSE 0 END)::BIGINT AS HC_N_RECHAZADAS,
           MAX(DAYS_DECISION)                                                  AS HC_DIAS_ULTIMA_DECISION
    FROM previous_application
    WHERE SK_ID_CURR IN (SELECT SK_ID_CURR FROM ids_train)
    GROUP BY SK_ID_CURR
),
-- ── POS_CASH_balance ─────────────────────────────────────────────────────────────────
pos_train AS (
    SELECT rowid AS fila, * FROM POS_CASH_balance
    WHERE MONTHS_BALANCE < 0 AND SK_ID_CURR IN (SELECT SK_ID_CURR FROM ids_train)
),
pos_agg AS (
    SELECT SK_ID_CURR,
           COUNT(DISTINCT SK_ID_PREV) AS HC_N_OPERACIONES_POS,
           MAX(SK_DPD)                AS HC_POS_MAX_ATRASO
    FROM pos_train
    GROUP BY SK_ID_CURR
),
pos_ultimo AS (                   -- último mes disponible de cada operación POS/Cash
    SELECT SK_ID_PREV, SK_ID_CURR, CNT_INSTALMENT_FUTURE AS CUOTAS_PENDIENTES_ULTIMO
    FROM pos_train
    QUALIFY ROW_NUMBER() OVER (PARTITION BY SK_ID_PREV ORDER BY MONTHS_BALANCE DESC, fila DESC) = 1
),
pos_ultimo_agg AS (
    SELECT SK_ID_CURR, MAX(CUOTAS_PENDIENTES_ULTIMO) AS HC_POS_MAX_CUOTAS_PENDIENTES
    FROM pos_ultimo
    GROUP BY SK_ID_CURR
),
-- ── credit_card_balance ──────────────────────────────────────────────────────────────
card_train AS (
    SELECT rowid AS fila, *,
           AMT_BALANCE / CASE WHEN AMT_CREDIT_LIMIT_ACTUAL > 0 THEN AMT_CREDIT_LIMIT_ACTUAL END AS UTILIZACION
    FROM credit_card_balance
    WHERE MONTHS_BALANCE < 0 AND SK_ID_CURR IN (SELECT SK_ID_CURR FROM ids_train)
),
card_agg AS (
    SELECT SK_ID_CURR,
           COUNT(DISTINCT SK_ID_PREV) AS HC_N_OPERACIONES_TARJETA,
           MAX(SK_DPD)                AS HC_CARD_MAX_ATRASO,
           MAX(UTILIZACION)           AS HC_CARD_MAX_UTILIZACION
    FROM card_train
    GROUP BY SK_ID_CURR
),
card_ultimo AS (                  -- saldo del último mes disponible de cada tarjeta
    SELECT SK_ID_PREV, SK_ID_CURR, AMT_BALANCE AS SALDO_ULTIMO_MES
    FROM card_train
    QUALIFY ROW_NUMBER() OVER (PARTITION BY SK_ID_PREV ORDER BY MONTHS_BALANCE DESC, fila DESC) = 1
),
card_ultimo_agg AS (
    SELECT SK_ID_CURR, MAX(SALDO_ULTIMO_MES) AS HC_CARD_MAX_ULTIMO_SALDO
    FROM card_ultimo
    GROUP BY SK_ID_CURR
),
-- ── installments_payments ────────────────────────────────────────────────────────────
pagos_train AS (                  -- cuotas anteriores (DAYS_INSTALMENT < 0); se conservan filas sin fecha de pago
    SELECT SK_ID_CURR, SK_ID_PREV,
           CASE WHEN DAYS_ENTRY_PAYMENT IS NOT NULL THEN 1 ELSE 0 END AS PAGO_REGISTRADO,
           CASE WHEN DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT < 0 THEN 0
                ELSE DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT END         AS DIAS_ATRASO   -- NULL si no hay fecha de pago
    FROM installments_payments
    WHERE DAYS_INSTALMENT < 0 AND SK_ID_CURR IN (SELECT SK_ID_CURR FROM ids_train)
),
pagos_agg AS (
    SELECT SK_ID_CURR,
           COUNT(*)                                                                AS HC_N_REGISTROS_PAGO,
           SUM(CASE WHEN PAGO_REGISTRADO = 1 AND DIAS_ATRASO > 0 THEN 1 ELSE 0 END)::BIGINT AS HC_N_PAGOS_TARDE,
           MAX(DIAS_ATRASO)                                                        AS HC_PAY_MAX_ATRASO,
           SUM(PAGO_REGISTRADO)::BIGINT                                            AS N_PAGOS_OBSERVADOS
    FROM pagos_train
    GROUP BY SK_ID_CURR
)
-- ── Cruce final: application_train + 5 resúmenes (LEFT JOIN) ─────────────────────────
SELECT a.*,
       b.BUREAU_N_CREDITOS, b.BUREAU_N_ACTIVOS, b.BUREAU_DEUDA_TOTAL, b.BUREAU_MAX_DIAS_ATRASO,
       b.BUREAU_MESES_OBSERVADOS, b.BUREAU_MESES_CON_ATRASO, b.BUREAU_MESES_ESTADO_CONOCIDO,
       b.BUREAU_MESES_CON_ATRASO / NULLIF(b.BUREAU_MESES_ESTADO_CONOCIDO, 0) AS BUREAU_PROP_MESES_CON_ATRASO,
       p.HC_N_SOLICITUDES, p.HC_N_APROBADAS, p.HC_N_RECHAZADAS, p.HC_DIAS_ULTIMA_DECISION,
       p.HC_N_RECHAZADAS / p.HC_N_SOLICITUDES                               AS HC_PROP_SOLICITUDES_RECHAZADAS,
       po.HC_N_OPERACIONES_POS, po.HC_POS_MAX_ATRASO, pu.HC_POS_MAX_CUOTAS_PENDIENTES,
       c.HC_N_OPERACIONES_TARJETA, c.HC_CARD_MAX_ATRASO, c.HC_CARD_MAX_UTILIZACION, cu.HC_CARD_MAX_ULTIMO_SALDO,
       i.HC_N_REGISTROS_PAGO, i.HC_N_PAGOS_TARDE, i.HC_PAY_MAX_ATRASO,
       i.HC_N_PAGOS_TARDE / NULLIF(i.N_PAGOS_OBSERVADOS, 0)                 AS HC_PROP_PAGOS_TARDE,
       CASE WHEN b.BUREAU_N_CREDITOS IS NOT NULL THEN 1 ELSE 0 END          AS TIENE_BUREAU,
       CASE WHEN p.HC_N_SOLICITUDES  IS NOT NULL THEN 1 ELSE 0 END          AS TIENE_HISTORIAL_HOME_CREDIT
FROM application_train a
LEFT JOIN bureau_agg      b  USING (SK_ID_CURR)
LEFT JOIN prev_agg        p  USING (SK_ID_CURR)
LEFT JOIN pos_agg         po USING (SK_ID_CURR)
LEFT JOIN pos_ultimo_agg  pu USING (SK_ID_CURR)
LEFT JOIN card_agg        c  USING (SK_ID_CURR)
LEFT JOIN card_ultimo_agg cu USING (SK_ID_CURR)
LEFT JOIN pagos_agg       i  USING (SK_ID_CURR)
ORDER BY a.SK_ID_CURR

Running query in 'DuckDBPyConnection'

Count
307511


Vista previa del tablón materializado (`LIMIT` porque tiene ~307 mil filas).

In [7]:
%%sql
SELECT * FROM tablon_general LIMIT 5

Running query in 'DuckDBPyConnection'

SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,BUREAU_N_CREDITOS,BUREAU_N_ACTIVOS,BUREAU_DEUDA_TOTAL,BUREAU_MAX_DIAS_ATRASO,BUREAU_MESES_OBSERVADOS,BUREAU_MESES_CON_ATRASO,BUREAU_MESES_ESTADO_CONOCIDO,BUREAU_PROP_MESES_CON_ATRASO,HC_N_SOLICITUDES,HC_N_APROBADAS,HC_N_RECHAZADAS,HC_DIAS_ULTIMA_DECISION,HC_PROP_SOLICITUDES_RECHAZADAS,HC_N_OPERACIONES_POS,HC_POS_MAX_ATRASO,HC_POS_MAX_CUOTAS_PENDIENTES,HC_N_OPERACIONES_TARJETA,HC_CARD_MAX_ATRASO,HC_CARD_MAX_UTILIZACION,HC_CARD_MAX_ULTIMO_SALDO,HC_N_REGISTROS_PAGO,HC_N_PAGOS_TARDE,HC_PAY_MAX_ATRASO,HC_PROP_PAGOS_TARDE,TIENE_BUREAU,TIENE_HISTORIAL_HOME_CREDIT
100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,None,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.08303696739132256,0.2629485927471776,0.13937578009978951,0.0247,0.0369,0.9722,0.6192,0.0143,0.0,0.069,0.0833,0.125,0.0369,0.0202,0.019,0.0,0.0,0.0252,0.0383,0.9722,0.6341,0.0144,0.0,0.069,0.0833,0.125,0.0377,0.022,0.0198,0.0,0.0,0.025,0.0369,0.9722,0.6243,0.0144,0.0,0.069,0.0833,0.125,0.0375,0.0205,0.0193,0.0,0.0,reg oper account,block of flats,0.0149,"Stone, brick",False,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,8,2,245781.0,0,108,27,71,0.38028169014084506,1,1,0,-606,0.0,1,0,6.0,None,None,None,None,19,0,0.0,0.0,1,1
100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003540999999999999,-16765,-1188,-1186.0,-291,None,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.3112673113812225,0.6222457752555098,None,0.0959,0.0529,0.9851,0.7959999999999999,0.0605,0.08,0.0345,0.2917,0.3333,0.013,0.0773,0.0549,0.0039,0.0098,0.092

## 4. Validación del tablón (antes de exportar)
*Grain* (una fila por `SK_ID_CURR`), conservación de información al unir, 148 columnas y cobertura de cada variable nueva.
Si alguna verificación falla, el notebook se detiene.

In [8]:
# ── Validación del tablón ─────────────────────────────────────────────
n_train = con.execute("SELECT COUNT(*) FROM application_train").fetchone()[0]
n_filas, n_clientes = con.execute("SELECT COUNT(*), COUNT(DISTINCT SK_ID_CURR) FROM tablon_general").fetchone()

cols_train = [r[1] for r in con.execute("PRAGMA table_info('application_train')").fetchall()]
cols_all   = [r[1] for r in con.execute("PRAGMA table_info('tablon_general')").fetchall()]
cols_nuevas = [c for c in cols_all if c not in cols_train]

target_tablon = con.execute("SELECT SUM(TARGET) FROM tablon_general").fetchone()[0]
target_train  = con.execute("SELECT SUM(TARGET) FROM application_train").fetchone()[0]

# Conservación: lo agregado en el tablón debe igualar lo que hay en las tablas hijas para clientes de train
bureau_tablon, = con.execute("SELECT SUM(BUREAU_N_CREDITOS) FROM tablon_general").fetchone()
bureau_fuente, = con.execute("SELECT COUNT(*) FROM bureau WHERE SK_ID_CURR IN (SELECT SK_ID_CURR FROM application_train)").fetchone()
prev_tablon, = con.execute("SELECT SUM(HC_N_SOLICITUDES) FROM tablon_general").fetchone()
prev_fuente, = con.execute("SELECT COUNT(*) FROM previous_application WHERE SK_ID_CURR IN (SELECT SK_ID_CURR FROM application_train)").fetchone()
rechazos_incoherentes, = con.execute("SELECT COUNT(*) FROM tablon_general WHERE HC_N_RECHAZADAS > HC_N_SOLICITUDES").fetchone()

checks = [
    ("Filas del tablón = filas de application_train",              n_filas == n_train,                      f"{n_filas:,} vs {n_train:,}"),
    ("Una fila por cliente (SK_ID_CURR único)",                     n_clientes == n_filas,                   f"{n_clientes:,} únicos de {n_filas:,}"),
    ("TARGET conservado (mismo número de defaults)",                target_tablon == target_train,           f"{target_tablon:,} vs {target_train:,}"),
    ("Sin nombres de columna duplicados",                           len(cols_all) == len(set(cols_all)),     f"{len(cols_all)} columnas"),
    ("Todas las columnas originales están presentes",               set(cols_train) <= set(cols_all),        f"{len(cols_train)} originales + {len(cols_nuevas)} nuevas"),
    ("148 columnas (122 + 8 + 16 + 2)",                             len(cols_all) == 148,                    f"{len(cols_all)} columnas"),
    ("Conservación bureau (suma de créditos = filas fuente)",       bureau_tablon == bureau_fuente,          f"{bureau_tablon:,} vs {bureau_fuente:,}"),
    ("Conservación previous_application",                           prev_tablon == prev_fuente,              f"{prev_tablon:,} vs {prev_fuente:,}"),
    ("Rechazos <= solicitudes previas",                             rechazos_incoherentes == 0,              f"{rechazos_incoherentes} casos incoherentes"),
]
for desc, ok, detalle in checks:
    print(("OK    " if ok else "FALLA ") + desc + "  ->  " + detalle)

# Cobertura de las variables nuevas: % de clientes con valor no nulo
cobertura = con.execute("SELECT " + ", ".join(f"ROUND(100.0*COUNT({c})/COUNT(*),1) AS {c}" for c in cols_nuevas) + " FROM tablon_general").df().T
cobertura.columns = ["% no nulo"]
display(cobertura)

assert all(ok for _, ok, _ in checks), "La validación falló: revisa las verificaciones marcadas como FALLA antes de exportar."
VALIDACION_OK = True
print("\nValidación completa.")

OK    Filas del tablón = filas de application_train  ->  307,511 vs 307,511
OK    Una fila por cliente (SK_ID_CURR único)  ->  307,511 únicos de 307,511
OK    TARGET conservado (mismo número de defaults)  ->  24,825 vs 24,825
OK    Sin nombres de columna duplicados  ->  148 columnas
OK    Todas las columnas originales están presentes  ->  122 originales + 26 nuevas
OK    148 columnas (122 + 8 + 16 + 2)  ->  148 columnas
OK    Conservación bureau (suma de créditos = filas fuente)  ->  1,465,325 vs 1,465,325
OK    Conservación previous_application  ->  1,413,701 vs 1,413,701
OK    Rechazos <= solicitudes previas  ->  0 casos incoherentes


,% no nulo
BUREAU_N_CREDITOS,85.7
BUREAU_N_ACTIVOS,85.7
BUREAU_DEUDA_TOTAL,83.3
BUREAU_MAX_DIAS_ATRASO,85.7
BUREAU_MESES_OBSERVADOS,29.9
BUREAU_MESES_CON_ATRASO,29.9
BUREAU_MESES_ESTADO_CONOCIDO,29.9
BUREAU_PROP_MESES_CON_ATRASO,29.1
HC_N_SOLICITUDES,94.6
HC_N_APROBADAS,94.6



Validación completa.


## 5. Validación contra el tablón oficial (`HomeCredit_base.csv`)
Se carga el CSV oficial de la profesora (si está en `RAW_DIR`) y se compara con `tablon_general`:
filas, columnas (nombres y orden), conjunto y orden de `SK_ID_CURR`, y **todos los valores** de las 148 columnas
(numéricas con tolerancia relativa `1e-9`; texto exacto; nulos deben coincidir). Además, detalle de 5 `SK_ID_CURR` al azar.

In [9]:
RUTA_OFICIAL = ubicar("HomeCredit_base.csv")
assert RUTA_OFICIAL is not None, f"No se encontró HomeCredit_base.csv en {RAW_DIR}: la validación queda pendiente."
con.execute(f'''
    CREATE OR REPLACE TABLE oficial AS
    SELECT * FROM read_csv_auto('{RUTA_OFICIAL.as_posix()}', header=true, sample_size=-1)
''')

esq_t = con.execute("DESCRIBE tablon_general").df()[["column_name", "column_type"]]
esq_o = con.execute("DESCRIBE oficial").df()[["column_name", "column_type"]]
n_of = con.execute("SELECT COUNT(*) FROM oficial").fetchone()[0]
solo_sql, solo_of = con.execute('''
    SELECT (SELECT COUNT(*) FROM tablon_general WHERE SK_ID_CURR NOT IN (SELECT SK_ID_CURR FROM oficial)),
           (SELECT COUNT(*) FROM oficial WHERE SK_ID_CURR NOT IN (SELECT SK_ID_CURR FROM tablon_general))
''').fetchone()
orden_distinto, = con.execute('''
    SELECT COUNT(*) FROM tablon_general t JOIN oficial o ON t.rowid = o.rowid WHERE t.SK_ID_CURR <> o.SK_ID_CURR
''').fetchone()

print(f"Filas   : SQL {n_filas:,} | oficial {n_of:,}")
print(f"Columnas: SQL {len(esq_t)} | oficial {len(esq_o)} | mismos nombres y orden: {list(esq_t.column_name) == list(esq_o.column_name)}")
print(f"SK_ID_CURR solo en SQL: {solo_sql} | solo en oficial: {solo_of} | filas en distinto orden: {orden_distinto}")

Filas   : SQL 307,511 | oficial 307,511
Columnas: SQL 148 | oficial 148 | mismos nombres y orden: True
SK_ID_CURR solo en SQL: 0 | solo en oficial: 0 | filas en distinto orden: 0


Comparación valor a valor de las 148 columnas (unión por `SK_ID_CURR`).

In [10]:
NUMERICOS = ("TINYINT", "SMALLINT", "INTEGER", "BIGINT", "HUGEINT", "UTINYINT", "USMALLINT", "UINTEGER", "UBIGINT",
             "FLOAT", "DOUBLE", "DECIMAL")
tipo_t = dict(zip(esq_t.column_name, esq_t.column_type))
tipo_o = dict(zip(esq_o.column_name, esq_o.column_type))
es_num = lambda t: t.split("(")[0] in NUMERICOS


def expr_diferencias(c):
    t, o = f't."{c}"', f'o."{c}"'
    if es_num(tipo_t[c]) and es_num(tipo_o[c]):
        tt, oo = f"CAST({t} AS DOUBLE)", f"CAST({o} AS DOUBLE)"
        return (f"SUM(CASE WHEN ({t} IS NULL) <> ({o} IS NULL) THEN 1 WHEN {t} IS NULL THEN 0 "
                f"WHEN abs({tt} - {oo}) > 1e-9 * greatest(1.0, abs({oo})) THEN 1 ELSE 0 END)")
    return f"SUM(CASE WHEN CAST({t} AS VARCHAR) IS DISTINCT FROM CAST({o} AS VARCHAR) THEN 1 ELSE 0 END)"


COLUMNAS = list(esq_o.column_name)
fila = con.execute("SELECT " + ", ".join(f'{expr_diferencias(c)} AS "{c}"' for c in COLUMNAS)
                   + " FROM tablon_general t JOIN oficial o USING (SK_ID_CURR)").fetchone()
COMPARACION = pd.DataFrame({"tipo_sql": [tipo_t[c] for c in COLUMNAS], "tipo_oficial": [tipo_o[c] for c in COLUMNAS],
                            "n_diferencias": [int(v) for v in fila]}, index=COLUMNAS)
print(f"Columnas con alguna diferencia: {(COMPARACION['n_diferencias'] > 0).sum()} de {len(COMPARACION)}")
display(COMPARACION.loc[cols_nuevas])            # detalle de las 26 variables construidas
display(COMPARACION.loc[COMPARACION["n_diferencias"] > 0])

Columnas con alguna diferencia: 0 de 148


,tipo_sql,tipo_oficial,n_diferencias
BUREAU_N_CREDITOS,BIGINT,DOUBLE,0
BUREAU_N_ACTIVOS,BIGINT,DOUBLE,0
BUREAU_DEUDA_TOTAL,DOUBLE,DOUBLE,0
BUREAU_MAX_DIAS_ATRASO,BIGINT,DOUBLE,0
BUREAU_MESES_OBSERVADOS,BIGINT,DOUBLE,0
BUREAU_MESES_CON_ATRASO,BIGINT,DOUBLE,0
BUREAU_MESES_ESTADO_CONOCIDO,BIGINT,DOUBLE,0
BUREAU_PROP_MESES_CON_ATRASO,DOUBLE,DOUBLE,0
HC_N_SOLICITUDES,BIGINT,DOUBLE,0
HC_N_APROBADAS,BIGINT,DOUBLE,0


,tipo_sql,tipo_oficial,n_diferencias


Detalle de 5 `SK_ID_CURR` al azar (semilla fija): valores SQL vs oficial en las 26 variables construidas y comprobación de las 148.

In [11]:
ids5 = [r[0] for r in con.execute("SELECT SK_ID_CURR FROM oficial USING SAMPLE reservoir(5 ROWS) REPEATABLE (42)").fetchall()]
lista = ", ".join(map(str, ids5))
t5 = con.execute(f"SELECT * FROM tablon_general WHERE SK_ID_CURR IN ({lista}) ORDER BY SK_ID_CURR").df().set_index("SK_ID_CURR")
o5 = con.execute(f"SELECT * FROM oficial WHERE SK_ID_CURR IN ({lista}) ORDER BY SK_ID_CURR").df().set_index("SK_ID_CURR")
# columnas numéricas a float para comparar NaN con NaN (DuckDB devuelve enteros con <NA>)
t5, o5 = [d.astype({c: "float64" for c in d.columns if pd.api.types.is_numeric_dtype(d[c])}) for d in (t5, o5)]
print("SK_ID_CURR elegidos:", sorted(ids5))

lado_a_lado = pd.concat({"SQL": t5[cols_nuevas].T, "oficial": o5[cols_nuevas].T}, axis=1).swaplevel(axis=1).sort_index(axis=1, level=0)
with pd.option_context("display.max_columns", 20, "display.float_format", lambda v: f"{v:,.6g}"):
    display(lado_a_lado)

pd.testing.assert_frame_equal(t5, o5[t5.columns], check_dtype=False, check_exact=False, rtol=1e-9)
print("Los 5 SK_ID_CURR coinciden en las 148 columnas.")

VALIDACION_OFICIAL_OK = (n_filas == n_of and list(esq_t.column_name) == list(esq_o.column_name) and solo_sql == 0
                         and solo_of == 0 and orden_distinto == 0 and int(COMPARACION["n_diferencias"].sum()) == 0)
assert VALIDACION_OFICIAL_OK, "El tablón SQL no reproduce exactamente HomeCredit_base.csv (ver tablas de arriba)."
print("VERIFICADO: el tablón SQL reproduce exactamente HomeCredit_base.csv (307,511 × 148, mismos valores).")

SK_ID_CURR elegidos: [100960, 102001, 117852, 120837, 123178]


SK_ID_CURR                       100960           102001         117852  \
                                    SQL  oficial     SQL oficial    SQL   
BUREAU_N_CREDITOS                     3        3       4       4      1   
BUREAU_N_ACTIVOS                      1        1       2       2      0   
BUREAU_DEUDA_TOTAL                    0        0 102,069 102,069      0   
BUREAU_MAX_DIAS_ATRASO                0        0       0       0      0   
BUREAU_MESES_OBSERVADOS             NaN      NaN     NaN     NaN    NaN   
BUREAU_MESES_CON_ATRASO             NaN      NaN     NaN     NaN    NaN   
BUREAU_MESES_ESTADO_CONOCIDO        NaN      NaN     NaN     NaN    NaN   
BUREAU_PROP_MESES_CON_ATRASO        NaN      NaN     NaN     NaN    NaN   
HC_N_SOLICITUDES                      2        2       1       1      4   
HC_N_APROBADAS                        2        2       1       1      2   
HC_N_RECHAZADAS                       0        0       0       0      0   
HC_DIAS_ULTIMA_DECISION            -179     -179    -893    -893    -87   
HC_PROP_SOLICITUDES_RECHAZADAS        0        0       0       0      0   
HC_N_OPERACIONES_POS                  1        1       1       1      1   
HC_POS_MAX_ATRASO                     0        0       0       0      0   
HC_POS_MAX_CUOTAS_PENDIENTES          7        7       0       0      0   
HC_N_OPERACIONES_TARJETA              1        1     NaN     NaN      1   
HC_CARD_MAX_ATRASO                    0        0     NaN     NaN      0   
HC_CARD_MAX_UTILIZACION        0.976927 0.976927     NaN     NaN      0   
HC_CARD_MAX_ULTIMO_SALDO       30,928.4 30,928.4     NaN     NaN      0   
HC_N_REGISTROS_PAGO                  22       22      12      12     12   
HC_N_PAGOS_TARDE                      0        0       0       0      0   
HC_PAY_MAX_ATRASO                     0        0       0       0      0   
HC_PROP_PAGOS_TARDE                   0        0       0       0      0   
TIENE_BUREAU                          1        1       1       1      1   
TIENE_HISTORIAL_HOME_CREDIT           1        1       1       1      1   

SK_ID_CURR                                120837            123178          
                               oficial       SQL   oficial     SQL oficial  
BUREAU_N_CREDITOS                    1         3         3       3       3  
BUREAU_N_ACTIVOS                     0         1         1       2       2  
BUREAU_DEUDA_TOTAL                   0       NaN       NaN 276,903 276,903  
BUREAU_MAX_DIAS_ATRASO               0         0         0       0       0  
BUREAU_MESES_OBSERVADOS            NaN       NaN       NaN     NaN     NaN  
BUREAU_MESES_CON_ATRASO            NaN       NaN       NaN     NaN     NaN  
BUREAU_MESES_ESTADO_CONOCIDO       NaN       NaN       NaN     NaN     NaN  
BUREAU_PROP_MESES_CON_ATRASO       NaN       NaN       NaN     NaN     NaN  
HC_N_SOLICITUDES                     4         3         3       5       5  
HC_N_APROBADAS                       2         2         2       4       4  
HC_N_RECHAZADAS                      0         1         1       0       0  
HC_DIAS_ULTIMA_DECISION            -87    -2,226    -2,226    -442    -442  
HC_PROP_SOLICITUDES_RECHAZADAS       0  0.333333  0.333333       0       0  
HC_N_OPERACIONES_POS                 1         1         1       4       4  
HC_POS_MAX_ATRASO                    0         0         0       7       7  
HC_POS_MAX_CUOTAS_PENDIENTES         0         1         1       1       1  
HC_N_OPERACIONES_TARJETA             1         1         1       1       1  
HC_CARD_MAX_ATRASO                   0         0         0       0       0  
HC_CARD_MAX_UTILIZACION              0  0.892052  0.892052       0       0  
HC_CARD_MAX_ULTIMO_SALDO             0         0         0       0       0  
HC_N_REGISTROS_PAGO                 12       109       109      20      20  
HC_N_PAGOS_TARDE                     0         3         3       9       9  
HC_PAY_MAX_ATRASO                    0         2   

Los 5 SK_ID_CURR coinciden en las 148 columnas.
VERIFICADO: el tablón SQL reproduce exactamente HomeCredit_base.csv (307,511 × 148, mismos valores).


## 6. Exportación
Se exporta el tablón SQL a `data/tablon_general_sql.parquet` (no se versiona) y se relee para confirmar que quedó completo.
El artefacto versionado del proyecto es `artifacts/tablon_general.parquet` (convertido desde `HomeCredit_base.csv`), idéntico en valores según la sección 5.

In [12]:
# ── Exportación del tablón ────────────────────────────────────────────
assert VALIDACION_OK, "Ejecuta primero la celda de validación."

RUTA_TABLON = REPO_DIR / "data" / "tablon_general_sql.parquet"
con.execute(f"COPY (SELECT * FROM tablon_general) TO '{RUTA_TABLON.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")

# Verificación posterior: se relee el archivo y se compara con la tabla
n_arch, = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_TABLON.as_posix()}')").fetchone()
n_cols_arch = len(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{RUTA_TABLON.as_posix()}')").fetchall())

print(f"Archivo : {RUTA_TABLON.relative_to(REPO_DIR).as_posix()}")
print(f"Tamaño  : {RUTA_TABLON.stat().st_size / 1e6:,.1f} MB")
print(f"Filas   : {n_arch:,}  (tablón: {n_filas:,})")
print(f"Columnas: {n_cols_arch}  (tablón: {len(cols_all)})")
assert n_arch == n_filas and n_cols_arch == len(cols_all), "El archivo no coincide con el tablón."
print("Exportación verificada.")
con.close()

Archivo : data/tablon_general_sql.parquet
Tamaño  : 25.8 MB
Filas   : 307,511  (tablón: 307,511)
Columnas: 148  (tablón: 148)
Exportación verificada.


## Anexo — Variables propias de la versión anterior (fuera del tablón oficial)

> **PENDIENTE DE DECISIÓN (Isaac):** mi versión anterior del tablón (133 columnas) incluía estas agregaciones, que **no** están
> en la lógica oficial y **no** forman parte de las 148 columnas. No se borran: quedan aquí, sin ejecutar, para decidir si se conservan
> (por ejemplo, como variables adicionales en una tabla aparte) o se descartan.

| Variable anterior | Qué hacía | Equivalente oficial más cercano |
|---|---|---|
| `bureau_credito_prom` | `AVG(AMT_CREDIT_SUM)` de los créditos en buró | ninguno (el oficial usa `SUM(AMT_CREDIT_SUM_DEBT)`) |
| `pos_n_registros` | nº de filas mensuales de POS/Cash | ninguno (el oficial cuenta operaciones: `HC_N_OPERACIONES_POS`) |
| `inst_dias_atraso_prom` | promedio de días de atraso **por cuota** (agregando pagos parciales por `SK_ID_PREV`, versión y nº de cuota) | ninguno (el oficial usa el **máximo** y por registro: `HC_PAY_MAX_ATRASO`) |
| `inst_deficit_total` | Σ (cuota − pagado) por cuota | ninguno |
| `cc_saldo_prom` | `AVG(AMT_BALANCE)` de tarjetas | ninguno (el oficial usa el saldo del último mes: `HC_CARD_MAX_ULTIMO_SALDO`) |

Criterio propio adicional: **agregar `installments_payments` a nivel cuota** (`SK_ID_PREV`, `NUM_INSTALMENT_VERSION`, `NUM_INSTALMENT_NUMBER`)
antes de llevarlo a `SK_ID_CURR`. El oficial trabaja **por registro de pago** (un pago parcial cuenta como registro aparte).

Diferencias que sí se corrigieron (las obliga la lógica oficial): filtros `MONTHS_BALANCE < 0` (bureau_balance, POS, tarjetas) y
`DAYS_INSTALMENT < 0` (installments), que la versión anterior no aplicaba; nombres en mayúsculas del diccionario oficial.

```sql
-- Versión anterior (sin ejecutar)
WITH bureau_extra AS (
    SELECT SK_ID_CURR, AVG(AMT_CREDIT_SUM) AS bureau_credito_prom
    FROM bureau GROUP BY SK_ID_CURR
),
pos_extra AS (
    SELECT SK_ID_CURR, COUNT(*) AS pos_n_registros
    FROM POS_CASH_balance GROUP BY SK_ID_CURR
),
cuota_agg AS (                    -- nivel cuota (una fila por cuota)
    SELECT SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER,
           MAX(AMT_INSTALMENT)  AS cuota,
           SUM(AMT_PAYMENT)     AS pagado,
           MAX(DAYS_ENTRY_PAYMENT) - MAX(DAYS_INSTALMENT) AS dias_atraso
    FROM installments_payments
    GROUP BY SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER
),
inst_extra AS (
    SELECT SK_ID_CURR,
           AVG(dias_atraso)                   AS inst_dias_atraso_prom,
           SUM(cuota - COALESCE(pagado, 0))   AS inst_deficit_total
    FROM cuota_agg GROUP BY SK_ID_CURR
),
cc_extra AS (
    SELECT SK_ID_CURR, AVG(AMT_BALANCE) AS cc_saldo_prom
    FROM credit_card_balance GROUP BY SK_ID_CURR
)
SELECT a.SK_ID_CURR, b.bureau_credito_prom, po.pos_n_registros,
       i.inst_dias_atraso_prom, i.inst_deficit_total, c.cc_saldo_prom
FROM application_train a
LEFT JOIN bureau_extra b  USING (SK_ID_CURR)
LEFT JOIN pos_extra    po USING (SK_ID_CURR)
LEFT JOIN inst_extra   i  USING (SK_ID_CURR)
LEFT JOIN cc_extra     c  USING (SK_ID_CURR)
```